# Cache aside
Read from cache first, then load and store missing data.


In [ ]:
# Cache-aside reads cache first, then fills it from the source of truth.
cache: dict[str, str] = {}
database = {"task:1": "Ship API"}

def get_task(key: str) -> str:
    if key not in cache:
        cache[key] = database[key]
    return cache[key]

print(get_task("task:1"))


## Polished version
Keep cache and provider behind interfaces, namespace keys, and treat cache failure as an optimization failure—not a chat failure.


In [ ]:
# Interfaces let production use Redis while tests use an in-memory cache.
import hashlib
from dataclasses import dataclass
from typing import Protocol

@dataclass(frozen=True)
class ChatRequest:
    prompt: str

@dataclass(frozen=True)
class ChatResponse:
    text: str

class Provider(Protocol):
    async def complete(self, request: ChatRequest) -> ChatResponse: ...

class Cache(Protocol):
    async def get(self, key: str) -> ChatResponse | None: ...
    async def set(self, key: str, value: ChatResponse) -> None: ...

class CacheUnavailable(Exception):
    pass

class FakeProvider:
    def __init__(self) -> None:
        self.calls = 0

    async def complete(self, request: ChatRequest) -> ChatResponse:
        self.calls += 1
        return ChatResponse(request.prompt.upper())

class MemoryCache:
    def __init__(self, available: bool = True) -> None:
        self.values: dict[str, ChatResponse] = {}
        self.available = available

    async def get(self, key: str) -> ChatResponse | None:
        if not self.available:
            raise CacheUnavailable
        return self.values.get(key)

    async def set(self, key: str, value: ChatResponse) -> None:
        if not self.available:
            raise CacheUnavailable
        self.values[key] = value

class ChatService:
    def __init__(self, provider: Provider, cache_store: Cache, namespace: str) -> None:
        self.provider = provider
        self.cache = cache_store
        self.namespace = namespace

    async def complete(self, request: ChatRequest) -> ChatResponse:
        # The provider/model namespace prevents collisions across deployments.
        material = f"{self.namespace}:{request.prompt}".encode()
        key = hashlib.sha256(material).hexdigest()
        try:
            cached = await self.cache.get(key)
        except CacheUnavailable:
            cached = None  # Fail open: the provider is still the source of truth.
        if cached is not None:
            return cached
        response = await self.provider.complete(request)
        try:
            await self.cache.set(key, response)
        except CacheUnavailable:
            pass  # Return the valid provider response even if caching fails.
        return response

provider = FakeProvider()
service = ChatService(provider, MemoryCache(available=False), "openai:gpt-demo")
print(await service.complete(ChatRequest("hello")), "provider calls:", provider.calls)


## Applied in this repository

The LLM [chat service](../00P2-project-llm-api/app/application/services.py) uses namespaced keys and continues through the provider when the [Redis adapter](../00P2-project-llm-api/app/infrastructure/cache.py) reports a cache failure.